# Burgers Equation – Numerical Tutorial
In the previous notebook (https://github.com/AGN000/CompressibleFlowCFDTutorial/blob/main/03_Stabilizing_Basic_FVM.ipynb) we stabilized Convection equation using Upwind and using artificial viscosity. Here we shall try those and study how much they can used for non-linear equation.



*"Today, I put equal stress on listing those problems where linear advection is **NOT** a guide to good practice."*
                            - **Phil Roe**,  (A Computational Autobiography)


This notebook demonstrates numerical solutions of the **inviscid Burgers equation** using several classical schemes.

The following methods are implemented:

1. Upwind Scheme
2. Godunov Scheme
3. Central Difference + Artificial Viscosity
4. Central Difference + Local Lax–Friedrichs (CD+LF)

The numerical solutions are compared with the **exact Riemann solution** and visualized using animations.

# 1. Governing Equation

The **inviscid Burgers equation** is a nonlinear conservation law

$$
\frac{\partial u}{\partial t} + \frac{\partial f(u)}{\partial x} = 0
$$

where the flux function is

$$
f(u) = \frac{u^2}{2}
$$

Substituting the flux gives

$$
\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0
$$

This equation is important because it contains the key nonlinear features of hyperbolic PDEs:

- nonlinear wave propagation  
- shock formation  
- rarefaction waves  

Despite its simplicity, it shares many characteristics with the **Euler equations of compressible flow**.

# 2. Riemann Initial Condition

We solve the **Riemann problem**, which starts with a discontinuous initial condition

$$
u(x,0)=
\begin{cases}
u_L, & x < 0 \\
u_R, & x > 0
\end{cases}
$$

Different choices of \(u_L\) and \(u_R\) generate different wave patterns.

## Test Cases Used

| Case | $u_L$ | $u_R$ | Wave Type |
|----|----|----|----|
| Rarefaction | -1 | 1 | Expanding fan |
| Shock | 1 | 0 | Moving discontinuity |
| Transonic | -1 | 0.5 | Rarefaction through zero |

# 3. Exact Solution of Burgers Riemann Problem

The exact solution depends on the relationship between \(u_L\) and \(u_R\).

## Rarefaction Wave ($u_L < u_R$)

A **rarefaction fan** forms and the solution becomes

$$
u(x,t)=
\begin{cases}
u_L & x/t < u_L \\
x/t & u_L \le x/t \le u_R \\
u_R & x/t > u_R
\end{cases}
$$

This represents a smooth expanding wave.

## Shock Wave ($u_L > u_R$)

A discontinuity propagates with **Rankine–Hugoniot speed**

$$
s = \frac{f(u_L)-f(u_R)}{u_L-u_R}
$$

For Burgers flux \(f(u)=u^2/2\)

$$
s = \frac{u_L+u_R}{2}
$$

The solution is

$$
u(x,t)=
\begin{cases}
u_L & x < st \\
u_R & x > st
\end{cases}
$$

# 4. Numerical Discretization

The spatial domain is discretized into \(N_x\) grid points

$$
x_i = x_{min} + i\Delta x
$$

where

$$
\Delta x = \frac{x_{max}-x_{min}}{N_x-1}
$$

Time integration uses a **CFL condition**

$$
\Delta t = CFL \frac{\Delta x}{\max |u|}
$$

This ensures numerical stability for hyperbolic PDEs.

# 5. Upwind Scheme

The conservative update formula is

$$
u_i^{n+1} =
u_i^n -
\frac{\Delta t}{\Delta x}
\left(
F_{i+\frac12} - F_{i-\frac12}
\right)
$$

The numerical flux is chosen based on the **direction of wave propagation**

$$
F_{i+\frac12} =
\begin{cases}
f(u_i) & u_i \ge 0 \\
f(u_{i+1}) & u_i < 0
\end{cases}
$$

This scheme is **first-order accurate** and introduces numerical diffusion.

# 6. Godunov Scheme (Correct upwind scheme)

Godunov's method solves a **local Riemann problem at each interface**.

The numerical flux is

$$
F_{i+\frac12} = F_{God}(u_L,u_R)
$$

For Burgers equation with convex flux

$$
F_{God}(u_L,u_R)
=
\max
\left(
f(\max(u_L,0)),
f(\min(u_R,0))
\right)
$$

This scheme is

- conservative  
- shock capturing  
- physically consistent

# 7. Central Difference + Artificial Viscosity

The convection term is approximated using a **central difference**

$$
\frac{\partial f}{\partial x}
\approx
\frac{f_{i+1}-f_{i-1}}{2\Delta x}
$$

To stabilize the scheme we add artificial viscosity

$$
\mu \frac{\partial^2 u}{\partial x^2}
$$

The second derivative is discretized as

$$
\frac{\partial^2 u}{\partial x^2}
\approx
\frac{u_{i+1}-2u_i+u_{i-1}}{\Delta x^2}
$$

The update becomes

$$
u_i^{n+1}
=
u_i^n
-
\Delta t
\left(
\frac{f_{i+1}-f_{i-1}}{2\Delta x}
\right)
+
\Delta t
\mu
\left(
\frac{u_{i+1}-2u_i+u_{i-1}}{\Delta x^2}
\right)
$$

# 8. Central Difference + Local Lax–Friedrichs (CD+LF)

A more robust method uses **local dissipation**.

The interface flux is

$$
F_{LLF}
=
\frac{1}{2}
\left(
f(u_L)+f(u_R)
\right)
-
\frac{1}{2}\alpha (u_R-u_L)
$$

where

$$
\alpha = \max(|u_L|,|u_R|)
$$

This dissipation adapts to the **local wave speed**.

# 9. Numerical Experiments

## Upwind scheme
Run for three Riemann cases

- Rarefaction
- Shock
- Transonic



In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (7,4)

# %%
nx = 400

xmin = -1
xmax = 1

x = np.linspace(xmin,xmax,nx)
dx = (xmax-xmin)/(nx-1)

CFL = 0.5
t_final = 0.5
# %%
def flux(u):
    return 0.5*u**2

# %%
def exact_solution(x,t,uL,uR):

    if t==0:
        return np.where(x<0,uL,uR)

    xi = x/(t+1e-12)
    u = np.zeros_like(x)

    for i in range(len(x)):

        if uL < uR:   # rarefaction

            if xi[i] < uL:
                u[i] = uL

            elif xi[i] > uR:
                u[i] = uR

            else:
                u[i] = xi[i]

        else:        # shock

            s = 0.5*(uL+uR)

            if x[i] < s*t:
                u[i] = uL
            else:
                u[i] = uR

    return u

# %%
cases = {
    "Rarefaction":(-1,1),
    "Shock":(1,0),
    "Transonic":(-1,0.5)
}


# %%
def solve_upwind(uL,uR):

    u = np.where(x<0,uL,uR)

    solutions=[]
    times=[]

    t=0

    while t<t_final:

        un = u.copy()

        a = np.max(np.abs(un))
        dt = CFL*dx/(a+1e-12)

        if t+dt>t_final:
            dt=t_final-t

        for i in range(1,nx-1):

            F_ip = flux(un[i]) if un[i]>=0 else flux(un[i+1])
            F_im = flux(un[i-1]) if un[i-1]>=0 else flux(un[i])

            u[i] = un[i] - dt/dx*(F_ip-F_im)

        u[0]=u[1]
        u[-1]=u[-2]

        t+=dt

        solutions.append(u.copy())
        times.append(t)

    return np.array(solutions),times

# %%
def animate_solution(solutions,times,uL,uR,title):

    fig,ax = plt.subplots()

    line_num, = ax.plot([],[],label="Numerical")
    line_ex,  = ax.plot([],[],'k--',label="Exact")

    ax.set_xlim(xmin,xmax)
    ax.set_ylim(-2,2)

    ax.legend()

    def update(frame):

        t = times[frame]

        line_num.set_data(x,solutions[frame])
        line_ex.set_data(x,exact_solution(x,t,uL,uR))

        ax.set_title(title + f"  t={t:.3f}")

        return line_num,line_ex

    anim = FuncAnimation(fig,update,frames=len(times),interval=40)

    return HTML(anim.to_jshtml())



# %%
for name,(uL,uR) in cases.items():

    sols,times = solve_upwind(uL,uR)

    display(animate_solution(sols,times,uL,uR,"Upwind : "+name))


### Godunov scheme
Represents the correct nonlinear upwinding.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ----------------------------------------------------------
# Flux Function (Burgers)
# ----------------------------------------------------------
def flux(u):
    return 0.5 * u**2

# ----------------------------------------------------------
# Exact Riemann Solver 
# ----------------------------------------------------------
def exact_solution(x, t, uL, uR):
    if t <= 1e-10:
        return np.where(x < 0, uL, uR)
    
    xi = x / t
    
    if uL > uR:  # Shock
        s = 0.5 * (uL + uR)
        return np.where(x < s * t, uL, uR)
    else:  # Rarefaction
        u = np.where(xi <= uL, uL,
            np.where(xi >= uR, uR, xi))  
        return u

# ----------------------------------------------------------
# Godunov Numerical Flux 
# ----------------------------------------------------------
def godunov_flux(uL, uR):

    f_uL_pos = flux(np.maximum(uL, 0))
    f_uR_neg = flux(np.minimum(uR, 0))
    return np.maximum(f_uL_pos, f_uR_neg)

# ----------------------------------------------------------
# Godunov Solver
# ----------------------------------------------------------
def solve_godunov(uL, uR, nx=400, t_final=0.5, CFL=0.5):
    xmin, xmax = -1, 1
    x = np.linspace(xmin, xmax, nx)
    dx = (xmax - xmin) / (nx - 1)
    
    u = np.where(x < 0, uL, uR)
    solutions = [u.copy()]
    times = [0.0]
    t = 0.0

    while t < t_final:
        # Calculate time step based on max wave speed
        a = np.max(np.abs(u))
        dt = CFL * dx / (a + 1e-12)
        if t + dt > t_final: dt = t_final - t
        

        F = godunov_flux(u[:-1], u[1:])
        
        # Update internal cells
        # u[i]^n+1 = u[i]^n - dt/dx * (F_i - F_{i-1})
        u[1:-1] = u[1:-1] - (dt/dx) * (F[1:] - F[:-1])
        
        # Neumann Boundary Conditions
        u[0], u[-1] = u[1], u[-2]
        
        t += dt
        solutions.append(u.copy())
        times.append(t)

    return np.array(solutions), times, x

# ----------------------------------------------------------
# Animation Helper
# ----------------------------------------------------------
def animate_case(name, uL, uR):
    sols, times, x_grid = solve_godunov(uL, uR)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    line_num, = ax.plot([], [], 'r-', lw=2, label="Godunov (Numerical)")
    line_ex,  = ax.plot([], [], 'k--', lw=1.5, label="Exact Solution")
    
    ax.set_xlim(-1, 1)
    ax.set_ylim(min(uL, uR) - 0.5, max(uL, uR) + 0.5)
    ax.set_title(f"Burgers Godunov: {name}")
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

    def update(frame):
        t = times[frame]
        line_num.set_data(x_grid, sols[frame])
        line_ex.set_data(x_grid, exact_solution(x_grid, t, uL, uR))
        return line_num, line_ex

    anim = FuncAnimation(fig, update, frames=len(times), interval=30, blit=True)
    plt.close() # Prevents double display in notebooks
    return HTML(anim.to_jshtml())

# ----------------------------------------------------------
# Run Test Cases
# ----------------------------------------------------------
cases = {
    "Rarefaction": (-1, 1),
    "Shock": (1, 0),
    "Transonic Rarefaction": (-1, 0.5)
}





### Shock animation for Gudnov Scheme

In [ ]:
animate_case("Shock", 1.0, 0.0)

### Expansion fan animation for Gudnov Scheme

In [ ]:
animate_case("Rarefaction", -1.0, 1.0)


### Transonic Rarefaction animation for Gudnov Scheme

In [ ]:
animate_case("Transonic Rarefaction", -1.0, 0.5)

### Central Difference + Viscosity
Works for some grids but may fail for others.



Code for nx = 400 and $\mu$ = 0.005

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ----------------------------------------------------------
# Parameters
# ----------------------------------------------------------
nx = 400
xmin, xmax = -1, 1
x = np.linspace(xmin, xmax, nx)
dx = (xmax - xmin) / (nx - 1)

CFL = 0.5
t_final = 0.5
mu = 0.005  

# ----------------------------------------------------------
# Functions
# ----------------------------------------------------------

def flux(u):
    return 0.5 * u**2

def exact_solution(x, t, uL, uR):
    if t <= 1e-10:
        return np.where(x < 0, uL, uR)
    
    xi = x / t
    if uL > uR:  # Shock
        s = 0.5 * (uL + uR)
        return np.where(x < s * t, uL, uR)
    else:        # Rarefaction
        return np.where(xi < uL, uL, np.where(xi > uR, uR, xi))

# ----------------------------------------------------------
# Solver (Vectorized CD + Viscosity)
# ----------------------------------------------------------

def solve_cd_visc(uL, uR):
    u = np.where(x < 0, uL, uR)
    solutions = [u.copy()]
    times = [0.0]
    t = 0.0

    while t < t_final:
        # 1. Stability: Convective dt (CFL) and Diffusive dt
        a = np.max(np.abs(u))
        dt_conv = CFL * dx / (a + 1e-12)
        dt_diff = 0.5 * dx**2 / (mu + 1e-12)
        dt = min(dt_conv, dt_diff)
        
        if t + dt > t_final:
            dt = t_final - t

        un = u.copy()
        
        # 2. Central Difference for Convection
        # F_{i+1/2} = 0.5 * (f(u_i) + f(u_{i+1}))
        f = flux(un)
        f_grad = (f[2:] - f[0:-2]) / (2 * dx)
        
        # 3. Central Difference for Diffusion (Second Derivative)
        # d2u/dx2 = (u_{i+1} - 2u_i + u_{i-1}) / dx^2
        u_diff = mu * (un[2:] - 2*un[1:-1] + un[0:-2]) / dx**2
        
        # Update interior points
        u[1:-1] = un[1:-1] - dt * f_grad + dt * u_diff

        # 4. Boundary Conditions (Neumann)
        u[0], u[-1] = u[1], u[-2]

        t += dt
        solutions.append(u.copy())
        times.append(t)

    return np.array(solutions), np.array(times)

# ----------------------------------------------------------
# Animation & Visualization
# ----------------------------------------------------------

def run_and_animate(name, uL, uR):
    sols, times = solve_cd_visc(uL, uR)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    line_num, = ax.plot([], [], 'r-', label="CD + Viscosity")
    line_ex,  = ax.plot([], [], 'k--', label="Exact (Inviscid)")
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(min(uL, uR) - 0.5, max(uL, uR) + 0.5)
    ax.legend()
    ax.grid(True, alpha=0.3)

    def update(frame):
        t = times[frame]
        line_num.set_data(x, sols[frame])
        line_ex.set_data(x, exact_solution(x, t, uL, uR))
        ax.set_title(f"Case: {name} (t={t:.3f})")
        return line_num, line_ex

    anim = FuncAnimation(fig, update, frames=len(times), interval=40, blit=True)
    plt.close()
    return HTML(anim.to_jshtml())

# ----------------------------------------------------------
# Run Test Cases
# ----------------------------------------------------------
cases = {
    "Rarefaction": (-1, 1),
    "Shock": (1, 0),
    "Transonic": (-1, 0.5)
}



### Shock animation for CD+dissipation Scheme

In [ ]:
run_and_animate("Shock", 1.0, 0.0)

### Expansion fan animation for CD+dissipation Scheme

In [ ]:
animate_case("Rarefaction", -1.0, 1.0)

### Transonic Rarefaction animation for CD+dissipation Scheme

In [ ]:
animate_case("Transonic Rarefaction", -1.0, 0.5)

Code for nx = 50 and $\mu$ = 0.005

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ----------------------------------------------------------
# Parameters
# ----------------------------------------------------------
nx = 50
xmin, xmax = -1, 1
x = np.linspace(xmin, xmax, nx)
dx = (xmax - xmin) / (nx - 1)

CFL = 0.5
t_final = 0.5
mu = 0.005  

# ----------------------------------------------------------
# Functions
# ----------------------------------------------------------

def flux(u):
    return 0.5 * u**2

def exact_solution(x, t, uL, uR):
     if t <= 1e-10:
        return np.where(x < 0, uL, uR)
    
    xi = x / t
    if uL > uR:  # Shock
        s = 0.5 * (uL + uR)
        return np.where(x < s * t, uL, uR)
    else:        # Rarefaction
        return np.where(xi < uL, uL, np.where(xi > uR, uR, xi))

# ----------------------------------------------------------
# Solver (Vectorized CD + Viscosity)
# ----------------------------------------------------------

def solve_cd_visc(uL, uR):
    u = np.where(x < 0, uL, uR)
    solutions = [u.copy()]
    times = [0.0]
    t = 0.0

    while t < t_final:
        # 1. Stability: Convective dt (CFL) and Diffusive dt
        a = np.max(np.abs(u))
        dt_conv = CFL * dx / (a + 1e-12)
        dt_diff = 0.5 * dx**2 / (mu + 1e-12)
        dt = min(dt_conv, dt_diff)
        
        if t + dt > t_final:
            dt = t_final - t

        un = u.copy()
        
        # 2. Central Difference for Convection
        # F_{i+1/2} = 0.5 * (f(u_i) + f(u_{i+1}))
        f = flux(un)
        f_grad = (f[2:] - f[0:-2]) / (2 * dx)
        
        # 3. Central Difference for Diffusion (Second Derivative)
        # d2u/dx2 = (u_{i+1} - 2u_i + u_{i-1}) / dx^2
        u_diff = mu * (un[2:] - 2*un[1:-1] + un[0:-2]) / dx**2
        
        # Update interior points
        u[1:-1] = un[1:-1] - dt * f_grad + dt * u_diff

        # 4. Boundary Conditions (Neumann)
        u[0], u[-1] = u[1], u[-2]

        t += dt
        solutions.append(u.copy())
        times.append(t)

    return np.array(solutions), np.array(times)

# ----------------------------------------------------------
# Animation & Visualization
# ----------------------------------------------------------

def run_and_animate(name, uL, uR):
    sols, times = solve_cd_visc(uL, uR)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    line_num, = ax.plot([], [], 'r-', label="CD + Viscosity")
    line_ex,  = ax.plot([], [], 'k--', label="Exact (Inviscid)")
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(min(uL, uR) - 0.5, max(uL, uR) + 0.5)
    ax.legend()
    ax.grid(True, alpha=0.3)

    def update(frame):
        t = times[frame]
        line_num.set_data(x, sols[frame])
        line_ex.set_data(x, exact_solution(x, t, uL, uR))
        ax.set_title(f"Case: {name} (t={t:.3f})")
        return line_num, line_ex

    anim = FuncAnimation(fig, update, frames=len(times), interval=40, blit=True)
    plt.close()
    return HTML(anim.to_jshtml())

# ----------------------------------------------------------
# Run Test Cases
# ----------------------------------------------------------
cases = {
    "Rarefaction": (-1, 1),
    "Shock": (1, 0),
    "Transonic": (-1, 0.5)
}

# Example execution

### Shock animation for CD+dissipation Scheme (Need viscosity tuning for each case)

In [ ]:
run_and_animate("Shock", 1.0, 0.0)

### Central Difference + Local Lax–Friedrichs
Provides improved robustness across different grid resolutions.

In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (7, 4)

# %%
# Grid Configuration
nx = 400
xmin, xmax = -1, 1
x = np.linspace(xmin, xmax, nx)
dx = (xmax - xmin) / (nx - 1)

CFL = 0.5
t_final = 0.5

# %%
def flux(u):
    return 0.5 * u**2

# %%
def exact_solution(x, t, uL, uR):
    """Exact Riemann solver for Burgers equation."""
    if t < 1e-10:
        return np.where(x < 0, uL, uR)
    xi = x / t
    if uL < uR:  # Rarefaction
        return np.where(xi <= uL, uL, np.where(xi >= uR, uR, xi))
    else:        # Shock (Rankine-Hugoniot speed: s = (uL + uR) / 2)
        s = 0.5 * (uL + uR)
        return np.where(x < s * t, uL, uR)

# %%
def solve_cd_lf(uL, uR):
     u = np.where(x < 0, uL, uR)
    solutions = [u.copy()]
    times = [0.0]
    t = 0.0

    while t < t_final:
        un = u.copy()

        # Adaptive time step based on max wave speed (CFL condition)
        a_max = np.max(np.abs(un))
        dt = CFL * dx / (a_max + 1e-12)
        if t + dt > t_final:
            dt = t_final - t

        # Interface states (nx-1 interfaces)
        uL_i = un[:-1]   # left state at interface i+1/2
        uR_i = un[1:]    # right state at interface i+1/2

        # Local max wave speed at each interface
        alpha = np.maximum(np.abs(uL_i), np.abs(uR_i))

        # Local Lax-Friedrichs numerical flux
        # F = 0.5*(f(uL) + f(uR)) - 0.5*alpha*(uR - uL)
        F = 0.5 * (flux(uL_i) + flux(uR_i)) - 0.5 * alpha * (uR_i - uL_i)

        # Conservative update for interior cells
        # u[i]^{n+1} = u[i]^n - (dt/dx) * (F_{i+1/2} - F_{i-1/2})
        u[1:-1] = un[1:-1] - (dt / dx) * (F[1:] - F[:-1])

        # Neumann (zero-gradient) boundary conditions
        u[0]  = u[1]
        u[-1] = u[-2]

        t += dt
        solutions.append(u.copy())
        times.append(t)

    return np.array(solutions), times

# %%
def animate_solution(sols, times, uL, uR, title="Burgers CD+LF"):
    """
    Animates the numerical vs exact solution side-by-side.
    Returns an HTML animation for Jupyter display.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    line_num, = ax.plot([], [], 'r-',  lw=2,   label="CD+LF (Numerical)")
    line_ex,  = ax.plot([], [], 'k--', lw=1.5, label="Exact Solution")
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10,
                        verticalalignment='top')

    ax.set_xlim(xmin, xmax)
    y_margin = 0.5
    ax.set_ylim(min(uL, uR) - y_margin, max(uL, uR) + y_margin)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

    def update(frame):
        t = times[frame]
        line_num.set_data(x, sols[frame])
        line_ex.set_data(x, exact_solution(x, t, uL, uR))
        time_text.set_text(f"t = {t:.3f}")
        return line_num, line_ex, time_text

    anim = FuncAnimation(fig, update, frames=len(times), interval=30, blit=True)
    plt.close()  # Prevents double display in notebooks
    return HTML(anim.to_jshtml())

# %%
# Test Cases
cases = {
    "Rarefaction":          (-1.0,  1.0),
    "Shock":                ( 1.0,  0.0),
    "Transonic Rarefaction":(-1.0,  0.5),
}

# %%
# Run all cases
for name, (uL, uR) in cases.items():
    sols, times = solve_cd_lf(uL, uR)
    display(animate_solution(sols, times, uL, uR, title=f"CD+LF — {name}"))

# 10. Summary

Key observations:

1. Burgers equation captures nonlinear wave behavior.
2. Central difference schemes alone are unstable for shocks.
3. Simple Upwind methods will not work
4. Gudnov-upwind is the correct upwind for non-linear equations
5. Central + scalar dissipation can stabilize non-linear equation but we may not know the correct dissipation co-efficient prior. Most of the time we tune it 
5. Local Lax–Friedrichs provides a robust shock-capturing method for all grid. This is very dissipative.  

*The linear advection equation is a powerful teacher, but a poor master. As we move to the Burgers' equation, we find that the core concepts remain relevant, yet require careful refinement to handle nonlinearity and shock formation. We aren't discarding what we learned; we are evolving it.*